In [4]:
#import Pkg
#Pkg.add("MimiqCircuits")

In [5]:
using MimiqCircuits

[ Info: Precompiling MimiqCircuits [281e006c-9395-434b-8b58-3065901f9db3] (cache misses: incompatible header (2))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up

SYSTEM: caught exception of type :MethodError while trying to print a failed 

In [6]:
conn = connect("quentin.hay-kergrohenn@epita.fr", "EiThuo5quei12")

┌ Warning: This connection methods is discuraged. Please use `connect()`, `connect(url)` or `connect(token[, url])`, if possible.
└ @ MimiqLink ~/.julia/packages/MimiqLink/PvESc/src/mimiq.jl:454


MimiqConnection:
├── url: https://mimiq.qperfect.io
├── max time limit: 180 minutes
├── Default time limit is equal to max time limit: 180 minutes
└── status: open

In [7]:
function cat_state(n::Int)::Circuit
    circuit = Circuit()
    push!(circuit, GateH(), 1)
    for i in 1:(n-1)
        push!(circuit, GateCX(), i, i + 1)
    end

    return circuit
end

cat_state (generic function with 1 method)

In [8]:
function push_gen_op!(c::Circuit, gate, physical_control::Int, offset::Int, physical_targets...)
    gateCA = control(1, gate)
    
    push!(c, GateH(), physical_control)
    push!(c, gateCA, physical_control, physical_targets...)
    push!(c, GateH(), physical_control)
    push!(c, Measure(), physical_control, physical_control)
end

push_gen_op! (generic function with 1 method)

In [9]:
function triangle_surface_code()
    c = cat_state(3)
    gateZ2 = parallel(2, GateZ())
    gateX3 = parallel(3, GateX())
    for _ in 1:5
        push_gen_op!(c, gateZ2, 4, 3, 1:2...)
        push_gen_op!(c, gateZ2, 5, 3, 1:2...)
        push_gen_op!(c, gateX3, 6, 3, 1:3...)
    end

    return c
end


triangle_surface_code (generic function with 1 method)

In [10]:
tsf = triangle_surface_code()

6-qubit, 6-bit circuit with 63 instructions:
├── H @ q[1]
├── CX @ q[1], q[2]
├── CX @ q[2], q[3]
├── H @ q[4]
├── C(⨷ ² Z) @ q[4], q[1], q[2]
├── H @ q[4]
├── M @ q[4], c[4]
├── H @ q[5]
├── C(⨷ ² Z) @ q[5], q[1], q[2]
├── H @ q[5]
├── M @ q[5], c[5]
├── H @ q[6]
⋮   ⋮
├── M @ q[6], c[6]
├── H @ q[4]
├── C(⨷ ² Z) @ q[4], q[1], q[2]
├── H @ q[4]
├── M @ q[4], c[4]
├── H @ q[5]
├── C(⨷ ² Z) @ q[5], q[1], q[2]
├── H @ q[5]
├── M @ q[5], c[5]
├── H @ q[6]
├── C(⨷ ³ X) @ q[6], q[1], q[2], q[3]
├── H @ q[6]
└── M @ q[6], c[6]

In [11]:
#job_logical = execute(conn, decompose(tsf), algorithm="mps", nsamples=1000, label="logical")
#res_logical = getresult(conn, job_logical)

In [12]:
function range2(i::Int)
    return range(i, i+1)
end

function push_Xstabalizers!(
    c::Circuit,
    d::Int,
    qbits::UnitRange{Int},
    ancillas::UnitRange{Int},
)
    gateCX2 = control(1, parallel(2, GateX()))
    gateCX4 = control(1, parallel(4, GateX()))

    n = d^2
    idx = 1
    # Low XX stabilizers
    for i in 2:2:d
        push!(c, gateCX2, ancillas[idx], qbits[range2(i)]...)
        idx += 1
    end

    # XXXX stabilizers
    XXXXstab = true
    for i in 0:d - 2
        for j in 1:d - 1
            if XXXXstab
                push!(
                    c, gateCX4, ancillas[idx],
                    qbits[range2(i*d + j)]...,
                    qbits[range2((i + 1)*d + j)]...
                )
                idx += 1
            end
            XXXXstab = !XXXXstab
        end
        XXXXstab = !XXXXstab
    end

    # High XX stabilizers
    for i in n - d + 1:2:n - 1
        push!(c, gateCX2, ancillas[idx], qbits[range2(i)]...)
        idx += 1
    end
end

function push_Zstabalizers!(
    c::Circuit,
    d::Int,
    qbits::UnitRange{Int},
    ancillas::UnitRange{Int},
)
    gateCZ2 = control(1, parallel(2, GateZ()))
    gateCZ4 = control(1, parallel(4, GateZ()))

    n = d^2
    idx = div(d^2, 2) + 1
    # All ZZ/ZZZZ stabilizers
    Zstab = false
    for i in 0:d - 2
        # Right ZZ stabilizers
        if !Zstab
            push!(c, gateCZ2, ancillas[idx], qbits[i*d + 1], qbits[(i+1)*d + 1])
            idx += 1
        end
        
        # ZZZZ stabilizers
        for j in 1:d - 1
            if Zstab
                push!(
                    c, gateCZ4, ancillas[idx],
                    qbits[range2(i*d + j)]...,
                    qbits[range2((i + 1)*d + j)]...
                )
                idx += 1
            end
            Zstab = !Zstab
        end  
        
        Zstab = !Zstab
        # Left ZZ stabilizers
        if !Zstab
            push!(c, gateCZ2, ancillas[idx], qbits[(i+1) * d], qbits[(i+2) * d])
            idx += 1
        end
    end
end

function rsf_detection(
    c::Circuit,
    d::Int,
    qbits::UnitRange{Int},
    ancillas::UnitRange{Int},
    cbits::UnitRange{Int}
)
    push!(c, Reset(), ancillas)
    push!(c, GateH(), ancillas)

    push_Xstabalizers!(c, d, qbits, ancillas)
    push_Zstabalizers!(c, d, qbits, ancillas)

    push!(c, GateH(), ancillas)
    push!(c, Measure(), ancillas, cbits)
end

rsf_detection (generic function with 1 method)

In [13]:
function push_condition!(c::Circuit, gate, qbit_idx::Int, cbits::UnitRange, target_cbits...)
    push!(
        c,
        IfStatement(gate, BitString(length(cbits), [target_cbits...])),
        qbit_idx,
        cbits...
    )
end

function rsf_correction(c::Circuit, d::Int)
    e = div(d, 2)
    half = div(d^2, 2)
    first, last = nothing, nothing
    for i in 0:d - 1
        #println("Index: ", i)
        first = (i&~1 + 1) * e + 1
        last = i&1 == 0 ? first - e : first + e

        if i&1 == 0
            push_condition!(c, GateZ(), i * d + 1, 1:half, first)
        end
            #println(first)
        for j in 2:d - 1
            push_condition!(c, GateZ(), i * d + j, 1:half, first, last)
            #println((first, last))
            first, last = last, first + 1
        end
        #println(first)
        if i&1 == 1
            push_condition!(c, GateZ(), (i + 1) * d, 1:half, first)
        end
    end

    for i in 1:d - 1
        for j in 1:d
            d
        end
    end
end

function rotated_surface_code(
    c::Circuit,
    distance::Int = 3,
    data_qbits::UnitRange{Int} = 1:9,
    ancillas::UnitRange{Int} = 10:17,
    cbits::UnitRange{Int} = 1:9
)
    for _ in 1:distance
        rsf_detection(c, distance, data_qbits, ancillas, cbits)
        if distance == 3
            push!(c, IfStatement(GateZ(), BitString("1000")), data_qbits[3], cbits[1:4]...)
            push!(c, IfStatement(GateZ(), BitString("0100")), data_qbits[1], cbits[1:4]...)
            push!(c, IfStatement(GateZ(), BitString("1100")), data_qbits[2], cbits[1:4]...)

            push!(c, IfStatement(GateZ(), BitString("0010")), data_qbits[9], cbits[1:4]...)
            push!(c, IfStatement(GateZ(), BitString("1010")), [data_qbits[3], data_qbits[9]], cbits[1:4]...)
            push!(c, IfStatement(GateZ(), BitString("0110")), data_qbits[5], cbits[1:4]...)
            push!(c, IfStatement(GateZ(), BitString("1110")), [data_qbits[2], data_qbits[1]], cbits[1:4]...)

            push!(c, IfStatement(GateZ(), BitString("0001")), data_qbits[7], cbits[1:4]...)
            push!(c, IfStatement(GateZ(), BitString("1001")), [data_qbits[3], data_qbits[7]], cbits[1:4]...)
            push!(c, IfStatement(GateZ(), BitString("0101")), [data_qbits[1], data_qbits[7]], cbits[1:4]...)
            push!(c, IfStatement(GateZ(), BitString("1101")), [data_qbits[2], data_qbits[7]], cbits[1:4]...)
            
            push!(c, IfStatement(GateZ(), BitString("0011")), data_qbits[8], cbits[1:4]...)
            push!(c, IfStatement(GateZ(), BitString("1011")), [data_qbits[3], data_qbits[8]], cbits[1:4]...)
            push!(c, IfStatement(GateZ(), BitString("0111")), [data_qbits[5], data_qbits[7]], cbits[1:4]...)
            push!(c, IfStatement(GateZ(), BitString("1111")), [data_qbits[2], data_qbits[8]], cbits[1:4]...)
        else
            throw("unimplemented")
            #rsf_correction(qc, distance)
    end
end

LoadError: ParseError:
[90m# Error @ [0;0m]8;;file:///home/quentix/Documents/tmp/epita/quantum/mimiq/project/group_3/In[13]#73:4\[90mIn[13]:73:4[0;0m]8;;\
    end
end[48;2;120;70;70m[0;0m
[90m#  └ ── [0;0m[91mExpected `end`[0;0m

In [ ]:
function logical_indexes(i)
    """
    Return the ranges of data_qubits, ancillas and c_bits depending on which logical qubit we look at.
    >>> logical_indexes(1)
    (1:9, 10:17, 1:9)
    """
    data_per_logic = 9
    ancillas_per_logic = 8
    c_bits_per_logic = 9

    data_qubits = (i-1) * (data_per_logic + ancillas_per_logic) + 1 : (i-1) * (data_per_logic + ancillas_per_logic) + data_per_logic
    ancillas = (i-1) * (data_per_logic + ancillas_per_logic) + data_per_logic + 1 : (i) * (data_per_logic + ancillas_per_logic)
    c_bits = (i-1) * data_per_logic + 1 : (i) * data_per_logic

    return (data_qubits, ancillas, c_bits)
end

In [ ]:
function logical_measure(c, l_target)
    # Measure all qubits of the logical qubit, and set results in the associated classical bits
    data_qubits, _, c_bits = logical_indexes(l_target)
    if typeof(c) == typeof(Circuit())
        drawable = nothing
    else
        drawable = c[2]
        c = c[1]
    end
    for (qbit, cbit) in zip(data_qubits, c_bits)
        push!(c, Measure(), qbit, cbit)
    end
    if drawable != nothing
        push!(drawable, Measure(), l_target, l_target)
    end
end

In [ ]:
qc = Circuit()
#push!(qc, Reset(), 1:17)
rotated_surface_code(qc, 3)
logical_measure(qc, 1)

In [ ]:
push!(qc, IfStatement(GateID(), BitString("0000")), 1, 11:14...)

In [ ]:
draw(decompose(qc))

In [ ]:
job_logical = execute(conn, decompose(qc), nsamples=100, label="physical_corrected")
res_logical = getresult(conn, job_logical)

In [ ]:
function count_bitstrings(n, res)
    """
    Return the dictionary of results from a execution of a distance-3 surface code circuit
    translated into classical binary strings.
    (for each pack of 9 bits, even number of ones = 0, odd number of ones = 1)
    """
    # Initialize an empty dictionary to store the bitstrings and their counts
    bitstring_counts = Dict{String, Int64}()
    
    # Loop over the samples and count the bitstrings
    for (bs, val) in histsamples(res)
        logical_bitstring = ""
        for i in 1:n
            # Count number of 1 in the data_qubits of the ith qubit
            nb_ones = count(x -> x == '1', string(bs[(i-1)*9 + 1 : i*9]))

            if nb_ones & 1 == 0
                logical_bitstring *= "0" # even number of ones => 0
            else
                logical_bitstring *= "1" # odd number of ones => 1
            end
        end

        if !haskey(bitstring_counts, logical_bitstring)
            bitstring_counts[logical_bitstring] = val
        else
            bitstring_counts[logical_bitstring] += val
        end
    end
    
    return bitstring_counts
end

In [ ]:
count_bitstrings(1, res_logical)